In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_1samp, ttest_ind,binom
import statsmodels.api as sm
from statsmodels.formula.api import ols
import warnings

In [2]:
df = df = pd.read_csv("property.csv")


In [3]:
missing_values = df.isnull().sum()
duplicate_rows = df.duplicated().sum()

print("Missing values per column:\n")
print(missing_values)

print("\nDuplicate rows:", duplicate_rows)


Missing values per column:

Suburb              0
Address             0
Rooms               0
Type                0
Price               0
Method              0
SellerG             0
Date                0
Distance            0
Postcode            0
Bedroom2            0
Bathroom            0
Car                62
Landsize            0
BuildingArea     6450
YearBuilt        5375
CouncilArea      1369
Lattitude           0
Longtitude          0
Regionname          0
Propertycount       0
dtype: int64

Duplicate rows: 0


In [4]:
df.head().T

,0,1,2,3,4
Suburb,Abbotsford,Abbotsford,Abbotsford,Abbotsford,Abbotsford
Address,85 Turner St,25 Bloomburg St,5 Charles St,40 Federation La,55a Park St
Rooms,2,2,3,3,4
Type,h,h,h,h,h
Price,1480000.0,1035000.0,1465000.0,850000.0,1600000.0
Method,S,S,SP,PI,VB
SellerG,Biggin,Biggin,Biggin,Biggin,Nelson
Date,3/12/2016,4/02/2016,4/03/2017,4/03/2017,4/06/2016
Distance,2.5,2.5,2.5,2.5,2.5
Postcode,3067.0,3067.0,3067.0,3067.0,3067.0


In [5]:
# Handling missing values
# BuildingArea
building_median = df['BuildingArea'].median()
df['BuildingArea'] = df['BuildingArea'].fillna(building_median)

# YearBuilt
year_median = df['YearBuilt'].median()
df['YearBuilt'] = df['YearBuilt'].fillna(year_median)
# CouncilArea
council_mode = df['CouncilArea'].mode()[0]
df['CouncilArea'] = df['CouncilArea'].fillna(council_mode)

In [6]:
warnings.filterwarnings("ignore")

In [7]:
#1.For the suburb of Altona, it is postulated that a typical property sells for $800,000.
price_data = df['Price']
# H0 (Null Hypothesis): Mean property price in Altona = 800000 
# H1 (Alternative Hypothesis): Mean property price in Altona > 800000
t_stat, p_value = ttest_1samp(price_data, 800000)
p_value = p_value / 2
print("Sample Mean :", round(price_data.mean(), 2))
print("Hypothesized Mean : 800000")
print("T-Statistic :", round(t_stat, 4))
print("P-Value :", round(p_value, 6))
alpha = 0.05
if (p_value < alpha) and (t_stat > 0):
    print("Reject the Null Hypothesis (H0)")
    print("Conclusion:The typical property price in Altona has significantly increased above $800,000.")    
else:
    print("Fail to Reject the Null Hypothesis (H0)")
    print("Conclusion: There is not enough evidence to say that the typical property price in Altona has increased above $800,000.") 

Sample Mean : 1075684.08
Hypothesized Mean : 800000
T-Statistic : 50.2516
P-Value : 0.0
Reject the Null Hypothesis (H0)
Conclusion:The typical property price in Altona has significantly increased above $800,000.


In [8]:
#2.For the year 2016, is there any difference in the prices of properties sold in the summer months vs winter months?
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df_2016 = df[df['Date'].dt.year == 2016].copy()
df_2016['Month'] = df_2016['Date'].dt.month
winter_months = [10, 11, 12, 1, 2, 3]
summer_months = [4, 5, 6, 7, 8, 9]
winter_prices = df_2016[df_2016['Month'].isin(winter_months)]['Price'].dropna()
summer_prices = df_2016[df_2016['Month'].isin(summer_months)]['Price'].dropna()
# H0:No difference in property prices between summer and winter months
# H1:There is a difference in property prices between summer and winter months
t_stat, p_value = ttest_ind(
    summer_prices,
    winter_prices,
    equal_var=False
)
print("Summer Mean Price :",round(summer_prices.mean(), 2))
print("Winter Mean Price :",round(winter_prices.mean(), 2))
print("T-Statistic :", round(t_stat, 4))
print("P-Value :", round(p_value, 6))
alpha = 0.05
if p_value < alpha:
    print("\nReject the Null Hypothesis (H0)")
    print("Conclusion:There is a significant difference in property prices between summer and winter months in 2016.")
else:
    print("\nFail to Reject the Null Hypothesis (H0)")
    print("Conclusion: There is no significant difference in property prices between summer and winter months in 2016.")
   

Summer Mean Price : 1048054.73
Winter Mean Price : 1116647.59
T-Statistic : -3.9211
P-Value : 9e-05

Reject the Null Hypothesis (H0)
Conclusion:There is a significant difference in property prices between summer and winter months in 2016.


In [9]:
#3. For the suburb of Abbotsford, what is the probability that out of 10 properties sold,3 will not have a car parking space? 
abbotsford_df = df[df['Suburb'] == 'Abbotsford']
no_car = abbotsford_df[abbotsford_df['Car'] == 0]
p = len(no_car) / len(abbotsford_df)
print("Probability of property having NO car parking :", round(p, 3))
n = 10
x = 3
probability = binom.pmf(x, n, p)
print("properties will NOT have car parking space:")
print(round(probability, 3))

Probability of property having NO car parking : 0.268
properties will NOT have car parking space:
0.26


In [10]:
#4 In the suburb of Abbotsford, what are the chances of finding a property with 3 rooms?  
three_room_properties = abbotsford_df[abbotsford_df['Rooms'] == 3]
probability = (len(three_room_properties)/len(abbotsford_df))
print("Probability of finding a property with 3 rooms in Abbotsford:")
print(round(probability, 3))

Probability of finding a property with 3 rooms in Abbotsford:
0.357


In [11]:
#5.
two_bathroom_properties = abbotsford_df[abbotsford_df['Bathroom'] == 2]
probability = (len(two_bathroom_properties)/len(abbotsford_df))
print("Probability of finding a property with 2 bathrooms in Abbotsford:")
print(round(probability, 3))

Probability of finding a property with 2 bathrooms in Abbotsford:
0.339


In [12]:
#6.
richmond_df = df[df['Suburb'] == 'Richmond']
# H0 (Null Hypothesis):Average property price in Richmond = $1,000,000
# H1 (Alternative Hypothesis):Average property price in Richmond ≠ $1,000,000
t_stat, p_value = ttest_1samp(price_data,1000000)
alpha = 0.05
print("Sample Mean Price :",round(price_data.mean(), 2))
print("Claimed Mean Price : 1000000")
print("T-Statistic :",round(t_stat, 4))
print("P-Value :",round(p_value, 6))
if p_value < alpha:
    print("Decision:Reject the Null Hypothesis (H0)")
    print("Business Conclusion:There is sufficient statistical evidence to conclude that the average property price in Richmond is significantly different from $1,000,000.")
else:
    print("\nDecision:")
    print("Fail to Reject the Null Hypothesis (H0)")
    print("\nBusiness Conclusion:There is insufficient statistical evidence to conclude that the average property price in Richmond differs from $1,000,000.")


Sample Mean Price : 1075684.08
Claimed Mean Price : 1000000
T-Statistic : 13.7957
P-Value : 0.0
Decision:Reject the Null Hypothesis (H0)
Business Conclusion:There is sufficient statistical evidence to conclude that the average property price in Richmond is significantly different from $1,000,000.


In [13]:
#7
with_parking = df[df['Car'] > 0]['Price'].dropna()
without_parking = df[df['Car'] == 0]['Price'].dropna()
# H0 (Null Hypothesis):Average prices are equal for both groups
# H1 (Alternative Hypothesis):Properties with car parking have a higher average selling price
t_stat, p_value = ttest_ind(with_parking,without_parking,equal_var=False)
p_value = p_value / 2
alpha = 0.05
print("\nAverage Price WITH Parking :",round(with_parking.mean(), 2))
print("Average Price WITHOUT Parking :",round(without_parking.mean(), 2))
print("\nT-Statistic :",round(t_stat, 4))
print("P-Value :",round(p_value, 6))
print("""
This test is appropriate because:
1. We are comparing the mean prices of TWO independent groups.
2. Group 1 = Properties with parking
3. Group 2 = Properties without parking
4. Price is a numerical variable.
""")
if (p_value < alpha) and (t_stat > 0):
    print("Decision:")
    print("Reject the Null Hypothesis (H0)")
    print("\nInterpretation of P-Value:")
    print("The p-value is less than 0.05,indicating strong statistical evidence that properties with car parking sell at higher average prices.")    
    print("Business Implications for Developers:")
    print("""
            1. Including parking spaces can increase property value.
            2. Buyers may be willing to pay premium prices for parking.
            3. Developers should prioritize parking facilities in future residential projects.
            4. Parking availability can improve market competitiveness.
          """)
else:
    print("Decision:")
    print("Fail to Reject the Null Hypothesis (H0)")
    print("Interpretation of P-Value:")
    print("There is insufficient statistical evidence to conclude that car parking significantly increases average property prices.")
    print("Business Implications for Developers:Parking may not significantly impactproperty prices based on this dataset.Developers may focus on other premium features.")


Average Price WITH Parking : 1074443.92
Average Price WITHOUT Parking : 1079088.01

T-Statistic : -0.2722
P-Value : 0.392737

This test is appropriate because:
1. We are comparing the mean prices of TWO independent groups.
2. Group 1 = Properties with parking
3. Group 2 = Properties without parking
4. Price is a numerical variable.

Decision:
Fail to Reject the Null Hypothesis (H0)
Interpretation of P-Value:
There is insufficient statistical evidence to conclude that car parking significantly increases average property prices.
Business Implications for Developers:Parking may not significantly impactproperty prices based on this dataset.Developers may focus on other premium features.


In [14]:
#8
anova_df = df[['Suburb', 'Type', 'Price']].dropna()
top_suburbs = anova_df['Suburb'].value_counts().head(10).index
anova_df = anova_df[anova_df['Suburb'].isin(top_suburbs)]
# Factor 1 : Suburb -- H0: Suburb has no effect on property price
# Factor 2 : Property Type --H0: Property type has no effect on property price
# Interaction:H0: No interaction effect between suburb and property type
model = ols('Price ~ C(Suburb) + C(Type) + C(Suburb):C(Type)',data=anova_df).fit()
anova_table = sm.stats.anova_lm(model,typ=2)
print(anova_table)
alpha = 0.05
suburb_p = anova_table.loc['C(Suburb)', 'PR(>F)']
print("Suburb",suburb_p)
if suburb_p < alpha:
    print("1. Suburb significantly affects property prices.")
else:
    print("1. Suburb does NOT significantly affect property prices.")
# Type Effect
type_p = anova_table.loc['C(Type)', 'PR(>F)']
print("property type:",type_p)
if type_p < alpha:
    print("2. Property type significantly affects property prices.")
else:
    print("2. Property type does NOT significantly affect property prices.")
# Interaction Effect
interaction_p = anova_table.loc['C(Suburb):C(Type)','PR(>F)']
print("interaction:",interaction_p)
if interaction_p < alpha:
    print("3. There is a significant interaction between suburb and property type.")
else:
    print("3. There is NO significant interaction between suburb and property type.")
    

                         sum_sq      df           F         PR(>F)
C(Suburb)          2.021766e+14     9.0  145.988879  3.058266e-218
C(Type)            2.771113e+14     2.0  900.441912  1.856326e-289
C(Suburb):C(Type)  7.118466e+13    18.0   25.700725   1.508430e-78
Residual           3.534513e+14  2297.0         NaN            NaN
Suburb 3.058265594024175e-218
1. Suburb significantly affects property prices.
property type: 1.8563256320883644e-289
2. Property type significantly affects property prices.
interaction: 1.5084303350811783e-78
3. There is a significant interaction between suburb and property type.


In [15]:
#9
p_value_1 = 0.032
alpha = 0.05
print("P-Value :", p_value)
print("Significance Level (alpha) :", alpha)
if p_value < alpha:        
    print("Decision:Reject the Null Hypothesis (H0)")        
    print("Interpretation: The p-value is less than 0.05,which indicates that the difference between the two suburbs is statistically significant.")
    print("Business Interpretation: Business stakeholders can conclude thatproperty prices differ significantly between the two suburbs.This means pricing strategies,investment decisions, and market planningshould be handled separately for each suburb.")
else:        
    print("Decision: Fail to Reject the Null Hypothesis (H0)")        
    print("Interpretation:There is insufficient statistical evidenceto conclude that property prices differbetween the two suburbs.")
    print("Business Interpretation:The business may treat both suburbs similarlyin terms of pricing and market strategy.")

P-Value : 0.39273749926992996
Significance Level (alpha) : 0.05
Decision: Fail to Reject the Null Hypothesis (H0)
Interpretation:There is insufficient statistical evidenceto conclude that property prices differbetween the two suburbs.
Business Interpretation:The business may treat both suburbs similarlyin terms of pricing and market strategy.


In [16]:
#10
# Group 1:Properties with MORE THAN 2 bathrooms
premium_group = df[df['Bathroom'] > 2]['Price'].dropna()
# Group 2:Properties with 2 or fewer bathrooms
standard_group = df[df['Bathroom'] <= 2]['Price'].dropna()
# H0:Properties with >2 bathrooms do NOT command higher prices
# H1:Properties with >2 bathrooms command higher prices
t_stat, p_value = ttest_ind(premium_group, standard_group,equal_var=False)
p_value = p_value / 2
alpha = 0.05
print("Average Price (>2 Bathrooms) :",round(premium_group.mean(), 2))
print("Average Price (<=2 Bathrooms) :",round(standard_group.mean(), 2))
print("\nT-Statistic :",round(t_stat, 4))
print("P-Value :",round(p_value, 6))
if (p_value < alpha) and (t_stat > 0):        
    print("Decision:Reject the Null Hypothesis (H0)")        
    print("Recommendation to Policymakers: There is strong statistical evidencethat properties with more than2 bathrooms command premium prices.Policymakers and urban planners may:1. Encourage development of larger family homes.2. Support housing projects with enhanced amenities.3. Consider bathroom count as an indicator   of high-value residential demand.")
else:        
    print("\nDecision: Fail to Reject the Null Hypothesis (H0)")        
    print("\nRecommendation to Policymakers: There is insufficient statistical evidenceto conclude that properties with more than2 bathrooms command premium prices.Policy decisions should consideradditional property features beforeassuming premium valuation.")

Average Price (>2 Bathrooms) : 1882824.2
Average Price (<=2 Bathrooms) : 1007347.94

T-Statistic : 28.8002
P-Value : 0.0
Decision:Reject the Null Hypothesis (H0)
Recommendation to Policymakers: There is strong statistical evidencethat properties with more than2 bathrooms command premium prices.Policymakers and urban planners may:1. Encourage development of larger family homes.2. Support housing projects with enhanced amenities.3. Consider bathroom count as an indicator   of high-value residential demand.
